# EDA Hari 2 — Kerangka

**Pelatihan ML untuk Operasi PLTP · Hari 2 · 08.30–12.00**

Sebelum membuat model, kita harus **percaya pada data**. Kita periksa data dalam empat lapis:

| Lapis | Langkah | Pertanyaan |
|---|---|---|
| 1 · Kualitas | 1–4 | Apakah data bisa dipercaya? Apa yang harus diperbaiki? |
| 2 · Satu kolom | 5–6 | Seperti apa bulan yang "normal"? Mana yang ekstrem? |
| 3 · Hubungan | 7–8 | Perbandingan mana yang adil? Kolom mana yang mirip? |
| 4 · Waktu | 9–10 | Adakah pola bulanan? Apa penyebabnya? |
| Simpan | 11–12 | Simpan data bersih, tulis 5 kalimat temuan |

**Cara kerja tiap langkah:**
1. Salin **prompt** ke asisten AI (Copilot / Antigravity / OpenCode / Gemini).
2. Tempel kodenya di sel kosong, **baca dulu**, lalu jalankan.
3. Jalankan sel **Cek** — bandingkan angkanya dengan yang tertulis "(harus ...)".
4. Tulis **arti** angkanya. Bagian ini tidak bisa dikerjakan AI.

**Aturan:** satu prompt = satu langkah · `mentah` dan file di `../data/` tidak boleh diubah · catat prompt di `prompt-log.md`.

Versi lengkap (kunci jawaban): `eda-hari2-lengkap.ipynb`.

## 0 · Persiapan (sudah jadi — tinggal jalankan)

In [ ]:
# Persiapan: library, data, dan gaya grafik
import sys
sys.path.insert(0, "..")

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import gaya_grafik as gg

gg.terapkan()

mentah   = pd.read_csv("../data/produksi_bulanan.csv")   # data MENTAH — jangan diubah
gangguan = pd.read_csv("../data/gangguan.csv")
unit     = pd.read_csv("../data/unit_pembangkit.csv")

print("mentah  :", mentah.shape)
print("gangguan:", gangguan.shape)
print("unit    :", unit.shape)

---
# Lapis 1 · Kualitas data

## Langkah 1 · Satu baris itu apa?

**Prompt:**
> Tampilkan ukuran `mentah` (baris × kolom), lima baris pertama, dan tipe data tiap kolom. Jelaskan satu baris mewakili apa. **Jangan ubah `mentah`.**

In [ ]:
# Langkah 1 — tempel kode dari AI di sini, BACA dulu, lalu jalankan


In [ ]:
# Cek
print("Baris  :", len(mentah), "(seharusnya 6 unit × 180 bulan = 1080 — kenapa lebih?)")
print("Kolom  :", mentah.shape[1], "(harus 10)")

## Langkah 2 · Cari masalah (jangan diperbaiki dulu)

**Prompt:**
> Baca `AGENTS.md` dan `data/kamus_data.md`. Periksa `mentah`: (1) jumlah nilai kosong per kolom, (2) nilai unik `kode_unit` dan jumlahnya, (3) jumlah baris duplikat, (4) min dan max tiap kolom angka. **Jangan ubah apa pun.**

In [ ]:
# Langkah 2 — tempel kode dari AI di sini, BACA dulu, lalu jalankan


In [ ]:
# Cek
print("Kode unit     :", mentah["kode_unit"].nunique(), "(seharusnya 6)")
print("Duplikat      :", mentah.duplicated().sum())
print("Energi kosong :", mentah["energi_aktual_mwh"].isna().sum())

**Jawaban Anda** (klik dua kali untuk mengisi):

- Temuan mana yang **salah data**, dan mana yang mungkin **kejadian nyata**?
  - *Jawab:* …

## Langkah 3 · Perbaiki satu per satu

Mulai sekarang kita bekerja di `df` (salinan). `mentah` tetap utuh.

In [ ]:
df = mentah.copy()   # salinan kerja
print("df:", df.shape)

**3a · Kode unit.** Prompt:
> Pada `df`, hapus spasi di awal/akhir `kode_unit` lalu jadikan huruf besar. Cetak jumlah unit sebelum dan sesudah.

In [ ]:
# Langkah 3a — tempel kode dari AI di sini, BACA dulu, lalu jalankan


**3b · Duplikat — lihat dulu.** Prompt:
> Tampilkan baris duplikat di `df` (kedua salinannya, `keep=False`). **Jangan hapus dulu.**

In [ ]:
# Langkah 3b-lihat — tempel kode dari AI di sini, BACA dulu, lalu jalankan


Sudah dilihat dan setuju? Prompt:
> Hapus baris duplikat dari `df`, reset index, lalu cetak jumlah baris sebelum dan sesudah.

In [ ]:
# Langkah 3b-hapus — tempel kode dari AI di sini, BACA dulu, lalu jalankan


**3c · Tanggal.** Prompt:
> Ubah kolom `tanggal` di `df` menjadi tipe datetime. Tampilkan tanggal paling awal dan paling akhir.

In [ ]:
# Langkah 3c — tempel kode dari AI di sini, BACA dulu, lalu jalankan


In [ ]:
# Cek
print("Baris df     :", len(df), "(harus 1080)")
print("Jumlah unit  :", df["kode_unit"].nunique(), "(harus 6)")
print("Bulan ganda  :", df.duplicated(["tanggal", "kode_unit"]).sum(), "(harus 0)")
print("Baris mentah :", len(mentah), "(harus tetap 1081)")

## Langkah 4 · Nilai kosong: cari sebabnya dulu

**4a · Lihat barisnya.** Prompt:
> Tampilkan baris `df` yang `energi_aktual_mwh`-nya kosong, dengan kolom `tanggal`, `kode_unit`, `cf_persen`, `jam_operasi`. Apakah unitnya sebenarnya beroperasi? **Jangan isi dulu.**

In [ ]:
# Langkah 4a — tempel kode dari AI di sini, BACA dulu, lalu jalankan


**4b · Putuskan.** Pilih satu, lalu catat alasannya di `prompt-log.md`:

| Pilihan | Risiko |
|---|---|
| Hapus 12 baris | unit-bulan itu hilang dari semua analisis |
| Isi dengan 0 | mencatat 12 bulan "mati" yang tidak pernah terjadi |
| Isi dari CF × kapasitas × jam, beri penanda `energi_diisi` | angkanya perkiraan, bukan hasil ukur |

**Prompt** (sesuaikan dengan pilihan Anda):
> Isi `energi_aktual_mwh` yang kosong di `df` dengan `cf_persen/100 × kapasitas_mw × jam dalam bulan` (`kapasitas_mw` dari `unit`). Tambahkan kolom `energi_diisi` (True untuk baris yang diisi). Jangan hapus baris. Cetak jumlah baris dan sisa nilai kosong.

In [ ]:
# Langkah 4b — tempel kode dari AI di sini, BACA dulu, lalu jalankan


In [ ]:
# Cek
print("Baris df       :", len(df), "(harus 1080)")
print("Ditandai diisi :", df["energi_diisi"].sum(), "(harus 12)")
df.loc[df["energi_diisi"], ["tanggal", "kode_unit", "cf_persen", "energi_aktual_mwh", "energi_rencana_mwh"]].round(0)

**Jawaban Anda** (klik dua kali untuk mengisi):

- Kenapa mengisi dengan 0 itu salah untuk 12 baris ini?
  - *Jawab:* …
- Apa risikonya mengisi dari CF?
  - *Jawab:* …

---
# Lapis 2 · Satu kolom: CF

## Langkah 5 · Seperti apa bulan yang normal?

**Prompt:**
> Buat histogram `cf_persen` di `df` dengan bin 5 % (0–100), pakai `gaya_grafik` (`gg`). Cetak rata-rata, median, minimum, dan maksimum. Lalu buat tabel per `kode_unit`: n, rata-rata, median, minimum.

In [ ]:
# Langkah 5 — tempel kode dari AI di sini, BACA dulu, lalu jalankan


In [ ]:
# Cek
print("Rata-rata:", round(df["cf_persen"].mean(), 1), "(harus 83.4)")
print("Median   :", round(df["cf_persen"].median(), 1), "(harus 85.3)")

## Langkah 6 · Bulan ekstrem: salah data atau kejadian nyata?

**6a · Cari outlier.** Prompt:
> Hitung batas outlier `cf_persen` di `df` dengan aturan IQR (Q1 − 1,5×IQR dan Q3 + 1,5×IQR). Cetak kedua batas dan jumlah outlier. Simpan baris outlier ke variabel `outlier` (salinan, kolom `tanggal`, `kode_unit`, `cf_persen`). **Jangan hapus apa pun dari `df`.**

In [ ]:
# Langkah 6a — tempel kode dari AI di sini, BACA dulu, lalu jalankan


**6b · Cocokkan dengan catatan gangguan.** Prompt:
> Untuk setiap baris di `outlier`, cari gangguan di `gangguan` pada unit yang sama dan bulan yang berada di antara bulan `tanggal_mulai` dan `tanggal_selesai`. Tambahkan kolom `jenis` (jenis gangguan, gabung dengan koma) dan `ada_gangguan` (True/False). Pakai perulangan `for` yang mudah dibaca. Cetak berapa outlier yang punya gangguan, dan tampilkan outlier dengan CF < 50.

In [ ]:
# Langkah 6b — tempel kode dari AI di sini, BACA dulu, lalu jalankan


In [ ]:
# Cek
print("Jumlah outlier         :", len(outlier), "(harus 91)")
print("Punya catatan gangguan :", outlier["ada_gangguan"].sum(), "(harus 91)")
print("CF < 50 yang ada MO    :", outlier[outlier["cf_persen"] < 50]["jenis"].str.contains("MO").sum(), "(harus 17)")
print("Baris df               :", len(df), "(harus tetap 1080)")

**Jawaban Anda** (klik dua kali untuk mengisi):

- Outlier ini salah data atau kejadian nyata? Apa buktinya?
  - *Jawab:* …
- Apa yang terjadi pada model forced outage (Hari 3) kalau 91 baris ini dibuang?
  - *Jawab:* …
- Untuk manajer, angka mana yang lebih menggambarkan bulan biasa: rata-rata atau median?
  - *Jawab:* …

---
# Lapis 3 · Hubungan antar kolom

## Langkah 7 · Bandingkan yang sebanding

**7a · Energi atau CF?** Prompt:
> Gabungkan `df` dengan `unit` berdasarkan `kode_unit` (how="left") menjadi `df_unit`; cetak jumlah baris sebelum dan sesudah. Lalu hitung rata-rata `energi_aktual_mwh` dan `cf_persen` per unit. Berapa kali energi ARL dibanding FRS, dan berapa selisih CF-nya?

In [ ]:
# Langkah 7a — tempel kode dari AI di sini, BACA dulu, lalu jalankan


**7b · Jenis turbin — selalu tampilkan n.** Prompt:
> Dari `df_unit`, buat tabel per `jenis_turbin`: jumlah unit, n (unit-bulan), rata-rata dan median `cf_persen`.

In [ ]:
# Langkah 7b — tempel kode dari AI di sini, BACA dulu, lalu jalankan


In [ ]:
# Cek
print("Baris df_unit       :", len(df_unit), "(harus 1080)")
print("jenis_turbin kosong :", df_unit["jenis_turbin"].isna().sum(), "(harus 0)")

## Langkah 8 · Kolom mana yang "bicara hal yang sama"?

**Prompt:**
> Hitung korelasi antara `uap_aktual_ton`, `energi_aktual_mwh`, `jam_operasi`, dan `cf_persen` di `df` (2 desimal). Buat scatter plot uap vs energi dengan warna per unit dari `gg.warna_unit`.

In [ ]:
# Langkah 8 — tempel kode dari AI di sini, BACA dulu, lalu jalankan


In [ ]:
# Cek
korelasi = df[["uap_aktual_ton", "energi_aktual_mwh", "jam_operasi", "cf_persen"]].corr()
print("r uap–energi :", round(korelasi.loc["uap_aktual_ton", "energi_aktual_mwh"], 2), "(sekitar 0.99)")
print("r jam–CF     :", round(korelasi.loc["jam_operasi", "cf_persen"], 2), "(sekitar 0.92)")

**Jawaban Anda** (klik dua kali untuk mengisi):

- Apakah turbin Binary "lebih buruk"? Tulis satu kalimat yang **jujur**.
  - *Jawab:* …
- Uap vs energi r ≈ 0,99. Apakah ini penemuan? Apa artinya untuk memilih feature model?
  - *Jawab:* …

---
# Lapis 4 · Waktu

## Langkah 9 · Adakah pola bulanan?

**9a · Rata-rata vs median per bulan.** Prompt:
> Dari `df`, hitung rata-rata dan median `cf_persen` per `bulan` (1–12), simpan sebagai `per_bulan` dengan kolom `mean` dan `median`. Buat grafik garis dua garis itu.

In [ ]:
# Langkah 9a — tempel kode dari AI di sini, BACA dulu, lalu jalankan


In [ ]:
# Cek
print("Rata-rata Maret     :", per_bulan.loc[3, "mean"], "(harus 80.0)")
print("Rata-rata September :", per_bulan.loc[9, "mean"], "(harus 80.4)")
print("Median terendah     :", per_bulan["median"].min(), "(harus 84.5)")

**9b · Siapa yang menarik Maret turun?** Prompt:
> Tampilkan semua unit-bulan di bulan Maret dengan `cf_persen` di bawah 75 dari `df`, urutkan dari yang terendah.

In [ ]:
# Langkah 9b — tempel kode dari AI di sini, BACA dulu, lalu jalankan


## Langkah 10 · Kapan perawatan (MO) dimulai?

**Prompt:**
> Dari `gangguan`, buat tabel silang jumlah kejadian per bulan `tanggal_mulai` (baris) dan `jenis` (kolom). Buat diagram batang khusus jenis MO. Jangan ubah `gangguan`.

In [ ]:
# Langkah 10 — tempel kode dari AI di sini, BACA dulu, lalu jalankan


In [ ]:
# Cek
print("MO mulai di Maret     :", tabel_silang.loc[3, "MO"], "(harus 7)")
print("MO mulai di September :", tabel_silang.loc[9, "MO"], "(harus 5)")

**Jawaban Anda** (klik dua kali untuk mengisi):

- Rata-rata Maret turun, median tidak. Jelaskan dalam satu kalimat untuk manajer.
  - *Jawab:* …
- Apa artinya untuk Q1 (memilih jendela perawatan)?
  - *Jawab:* …

---
## Langkah 11 · Simpan data bersih

File ini dipakai di sesi siang (clustering).

**Prompt:**
> Simpan `df` ke `../data/olahan/produksi_bersih.csv` (buat folder bila belum ada). Sebelum menyimpan, tambahkan `assert`: 1.080 baris, 6 unit, tidak ada unit-bulan ganda.

In [ ]:
# Langkah 11 — tempel kode dari AI di sini, BACA dulu, lalu jalankan


In [ ]:
# Cek
bersih = pd.read_csv("../data/olahan/produksi_bersih.csv")
print("File bersih  :", len(bersih), "baris,", bersih["kode_unit"].nunique(), "unit (harus 1080, 6)")
print("energi_diisi :", "energi_diisi" in bersih.columns, "(harus True)")
print("CSV mentah   :", len(pd.read_csv("../data/produksi_bulanan.csv")), "baris (harus tetap 1081)")

## Langkah 12 · Lima kalimat temuan

Satu kalimat per lapis + satu untuk keputusan. **Setiap kalimat memuat angka.** Salin juga ke `prompt-log.md`.

1. **Kualitas:** …
2. **Satu kolom:** …
3. **Hubungan:** …
4. **Waktu:** …
5. **Untuk tim perawatan:** …

**Prompt — minta AI memeriksa (harus menghitung ulang):**
> Ini lima kalimat temuan EDA saya. Periksa setiap angkanya dengan menghitung ulang dari `df`, `gangguan`, dan `unit` — tulis kodenya. Tandai kalimat yang menyimpulkan lebih dari yang didukung data.
>
> [tempel lima kalimat Anda]

In [ ]:
# Langkah 12 — tempel kode pemeriksaan dari AI di sini


---
## Checkpoint 12.00

- [ ] `data/olahan/produksi_bersih.csv` ada: 1.080 baris, 6 unit, kolom `energi_diisi`
- [ ] notebook ini jalan dengan *Restart & Run All*
- [ ] lima kalimat temuan + keputusan nilai kosong ada di `prompt-log.md`

**Bisa menjelaskan:** kenapa 1.081 jadi 1.080 · kenapa nilai kosong tidak diisi 0 · kenapa outlier tidak dibuang · kenapa rata-rata Maret turun tapi median tidak.